In [0]:
# =============================================================================
# segmentation_snapshot  -  capture a PER-CASE baseline of what is segmented and where,
# so upcoming segmentation changes can be diffed case-by-case (ADDED/REMOVED/MOVED),
# not just as a moving total. One row per case.
#
# SAFETY (per Peter): this is a TEST-ONLY tool.
#   - READS the pipeline gold validation tables + stg_segmentation_states  (READ-ONLY).
#   - NEVER writes to any dev / pipeline / raw / proper-data table.
#   - Its ONLY writes are: (1) a clearly-named TEST table `test_segmentation_snapshots`
#     and (2) files under your Results folder. Both are gated by flags below.
#   - WRITE_TABLE defaults to FALSE so nothing lands in Databricks until the table
#     name/home is AGREED. Files-only is safe to run now.
#
# Pair with segmentation_diff. Run after a full pipeline run to lock a baseline.
# =============================================================================

In [0]:
# ---- CELL 0 : config + auth ----
from pyspark.sql import functions as F
from pyspark.sql.functions import *
import uuid, datetime

# --- AUTO identity: unique + datetime-stamped so this notebook RUNS UNMODIFIED and never collides ---
SNAP_ID        = str(uuid.uuid4())          # unique per run (guarantees no duplicate snapshot)
_NOW           = datetime.datetime.now()
SNAPSHOT_TAG   = "postfix1290_20260819"      # OPTIONAL context suffix, e.g. "pre2307" / "post2307"; leave "" for timestamp-only
DATA_CUT_LABEL = "snap-" + _NOW.strftime("%Y%m%d-%H%M%S") + (("-" + SNAPSHOT_TAG) if SNAPSHOT_TAG else "")
NOTE           = "segmentation per-case baseline"          # free text stored with the snapshot
SNAPSHOT_TABLE = "test_segmentation_snapshots"   # AGREED: same home as test_automation_runs2 (default DB). Clearly-TEST, never a dev table.
WRITE_TABLE    = True     # append-only; table CREATED on first run
WRITE_FILES    = True     # export xlsx + parquet to Results folder (safe)
SEG_TBL        = "hive_metastore.ariadm_active_appeals.stg_segmentation_states"   # best-effort gate fields

# clone CaseNos from the config (is_clone flag). Keep in sync with the clone notebooks.
CLONES = set(["DA/00092/2026","DA/00243/2026","DC/00031/2026","DC/00092/2026","EA/00554/2026",
 "EA/00746/2026","EA/01234/2026","EA/01319/2026","EA/01319/2027","EA/01321/2026","EA/01959/2026",
 "EA/01959/2027","EA/03117/2026","EA/03592/2026","EA/03593/2026","EA/06826/2026","EA/08372/2026",
 "EA/10544/2026","EA/11255/2026","EA/13092/2026","HU/00575/2026","HU/01174/2026","HU/02151/2026",
 "HU/02313/2026","HU/02313/2027","HU/13121/2026","LR/00060/2026","PA/00187/2026","PA/01034/2026",
 "PA/01306/2026","PA/01983/2026","PA/02147/2026","PA/04157/2026","RP/00009/2026"])
GOLD = {
 "appealSubmitted":"hive_metastore.appealsubmitted_gold.stg_main_appeal_submitted_validation",
 "paymentPending":"hive_metastore.paymentPending_gold.stg_main_paymentPending_validation",
 "awaitingRespondentEvidence(a)":"hive_metastore.awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation",
 "awaitingRespondentEvidence(b)":"hive_metastore.awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation",
 "caseUnderReview":"hive_metastore.caseunderreview_gold.stg_main_case_under_review_validation",
 "decided(a)":"hive_metastore.decideda_gold.stg_main_decided_a_validation",
 "decided(b)":"hive_metastore.decidedb_gold.stg_main_decided_b_validation",
 "decision":"hive_metastore.decision_gold.stg_main_decision_validation",
 "ended":"hive_metastore.ended_gold.stg_main_ended_validation",
 "ftpaDecided":"hive_metastore.ftpadecided_gold.stg_main_ftpadecided_validation",
 "ftpaSubmitted(a)":"hive_metastore.ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation",
 "ftpaSubmitted(b)":"hive_metastore.ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation",
 "listing":"hive_metastore.listing_gold.stg_main_listing_validation",
 "prepareForHearing":"hive_metastore.prepareforhearing_gold.stg_main_prepare_for_hearing_validation",
 "reasonsForAppealSubmitted":"hive_metastore.reasonsforappealsubmitted_gold.stg_main_reasons_for_appeal_submitted_validation",
 "remitted":"hive_metastore.remitted_gold.stg_main_remitted_validation",
}
# ARCHIVE appeals segmentation source (canonical, from Overall/1_OVERALL_RECONCILIATION_TESTS ARCHIVE_SEGMENTS):
# each = (Segment label, table with CaseNo + Segment). stg_appeals_filtered (FTA) is itself a union of sub-stages.
ARCHIVE_SEGMENTS = [
 ("FTA", "hive_metastore.ariadm_arm_fta.stg_appeals_filtered"),
 ("UTA", "hive_metastore.ariadm_arm_uta.stg_appeals_filtered"),
 ("FPA", "hive_metastore.ariadm_arm_fpa.stg_filepreservedcases_filtered"),
]
_c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")

In [0]:
# ---- CELL 1 : build the per-case segmented set (union of the 16 gold validation tables) ----
clone_bc = spark.sparkContext.broadcast(CLONES)
@udf("boolean")
def is_clone_udf(ref): return (ref in clone_bc.value) if ref is not None else False

parts=[]; skipped=[]
for st, tbl in GOLD.items():
    try:
        g=spark.table(tbl)
        key = "appealReferenceNumber" if "appealReferenceNumber" in g.columns else "CaseNo"
        sel=(g.withColumn("appealReferenceNumber", trim(col(key)))
               .withColumn("assigned_state", lit(st))
               .withColumn("is_valid", col("is_valid").cast("boolean") if "is_valid" in g.columns else lit(None).cast("boolean"))
               .select("appealReferenceNumber","assigned_state","is_valid"))
        parts.append(sel)
    except Exception as e:
        skipped.append((st, str(e).splitlines()[0][:100]))
if not parts: raise Exception("no gold validation tables readable: "+str(skipped))
cases = parts[0]
for p in parts[1:]: cases = cases.unionByName(p)

# enrich: is_clone + casePrefix + best-effort gate fields from stg_segmentation_states
cases = (cases
    .withColumn("is_clone", is_clone_udf(col("appealReferenceNumber")))
    .withColumn("casePrefix", upper(split(col("appealReferenceNumber"),"/").getItem(0))))

def pick(cols, *names):
    return next((c for c in cols if c.lower() in [n.lower() for n in names]), None)
try:
    seg=spark.table(SEG_TBL); sc=seg.columns
    ct=pick(sc,"CaseType"); dp=pick(sc,"DeptId","DepartmentId"); kd=pick(sc,"KeyDate"); dd=pick(sc,"DecisionDate")
    segsel=seg.withColumn("_ref", trim(col("CaseNo"))) if "CaseNo" in sc else None
    if segsel is not None:
        segsel=segsel.select("_ref",
            *( [col(ct).cast("string").alias("caseType")] if ct else [lit(None).cast("string").alias("caseType")] ),
            *( [col(dp).cast("string").alias("deptId")] if dp else [lit(None).cast("string").alias("deptId")] ),
            *( [col(kd).cast("string").alias("keyDate")] if kd else [lit(None).cast("string").alias("keyDate")] ),
            *( [col(dd).cast("string").alias("decisionDate")] if dd else [lit(None).cast("string").alias("decisionDate")] ),
        ).dropDuplicates(["_ref"])
        cases=cases.join(segsel, cases.appealReferenceNumber==segsel._ref, "left").drop("_ref")
    else:
        for cnm in ["caseType","deptId","keyDate","decisionDate"]: cases=cases.withColumn(cnm, lit(None).cast("string"))
except Exception as e:
    for cnm in ["caseType","deptId","keyDate","decisionDate"]: cases=cases.withColumn(cnm, lit(None).cast("string"))
    skipped.append(("SEG_TBL enrich", str(e).splitlines()[0][:100]))

# terminal (latest) status per case = the CaseStatus/Outcome that DRIVES segmentation
# (this is the 't.' the seg query keys on -> explains 2307 decided(a)<->ended moves).
from pyspark.sql.window import Window
STATUS_TBL = "hive_metastore.ariadm_active_appeals_bronze.bronze_status_htype_clist_list_ltype_court_lsitting_adj"
try:
    stt = spark.table(STATUS_TBL).withColumn("_cn", trim(col("CaseNo"))); sc2 = stt.columns
    _cs=pick(sc2,"CaseStatus"); _oc=pick(sc2,"Outcome"); _sid=pick(sc2,"StatusId"); _dd=pick(sc2,"DecisionDate")
    if _cs and _oc and _sid:
        _w = Window.partitionBy("_cn").orderBy(col(_sid).desc())
        term = (stt.withColumn("_rn", row_number().over(_w)).filter(col("_rn")==1)
                   .select(col("_cn").alias("_sref"), col(_cs).cast("string").alias("caseStatus"),
                           col(_oc).cast("string").alias("outcome"),
                           (col(_dd).cast("string") if _dd else lit(None).cast("string")).alias("statusDecisionDate")))
        cases = cases.join(term, cases.appealReferenceNumber==term._sref, "left").drop("_sref")
    else:
        for c in ["caseStatus","outcome","statusDecisionDate"]: cases=cases.withColumn(c, lit(None).cast("string"))
except Exception as e:
    for c in ["caseStatus","outcome","statusDecisionDate"]: cases=cases.withColumn(c, lit(None).cast("string"))
    skipped.append(("STATUS enrich", str(e).splitlines()[0][:100]))

# snapshot meta (SNAP_ID + _NOW come from CELL 0 -> auto/unique, runs unmodified)
NOW = _NOW
SNAP_COLS = ["snapshot_id","snapshot_datetime","data_cut_label","note","tier",
             "appealReferenceNumber","assigned_state","is_valid","is_clone","casePrefix",
             "caseStatus","outcome","statusDecisionDate","caseType","deptId","keyDate","decisionDate"]
snap = (cases
    .withColumn("snapshot_id", lit(SNAP_ID))
    .withColumn("snapshot_datetime", lit(NOW).cast("timestamp"))
    .withColumn("data_cut_label", lit(DATA_CUT_LABEL))
    .withColumn("note", lit(NOTE))
    .withColumn("tier", lit("active"))
    .select(*SNAP_COLS))

# ---- ARCHIVE arm: union the canonical FTA/UTA/FPA segment tables (CaseNo + Segment) ----
arch_parts = []
for seg_label, tbl in ARCHIVE_SEGMENTS:
    try:
        a = spark.table(tbl); acols = a.columns
        cc = pick(acols, "CaseNo"); sg = pick(acols, "Segment")
        if not cc:
            skipped.append((f"ARCHIVE {seg_label}", f"no CaseNo in {tbl}")); continue
        ar = (a.withColumn("appealReferenceNumber", trim(col(cc)))
               .withColumn("assigned_state", (col(sg).cast("string") if sg else lit("ARIA"+seg_label)))
               .select("appealReferenceNumber","assigned_state").dropDuplicates(["appealReferenceNumber"]))
        arch_parts.append(ar)
    except Exception as e:
        skipped.append((f"ARCHIVE {seg_label}", str(e).splitlines()[0][:100]))
if arch_parts:
    arch = arch_parts[0]
    for p in arch_parts[1:]: arch = arch.unionByName(p)
    arch = (arch.dropDuplicates(["appealReferenceNumber"])
        .withColumn("is_valid", lit(None).cast("boolean"))
        .withColumn("is_clone", is_clone_udf(col("appealReferenceNumber")))
        .withColumn("casePrefix", upper(split(col("appealReferenceNumber"),"/").getItem(0)))
        .withColumn("snapshot_id", lit(SNAP_ID)).withColumn("snapshot_datetime", lit(NOW).cast("timestamp"))
        .withColumn("data_cut_label", lit(DATA_CUT_LABEL)).withColumn("note", lit(NOTE))
        .withColumn("tier", lit("archive")))
    for c in ["caseStatus","outcome","statusDecisionDate","caseType","deptId","keyDate","decisionDate"]:
        arch = arch.withColumn(c, lit(None).cast("string"))
    snap = snap.unionByName(arch.select(*SNAP_COLS))

snap = snap.cache()
TOTAL = snap.count()

In [0]:
# ---- CELL 2 : write central table + files ----
msg=[]
if WRITE_TABLE:
    snap.write.option("mergeSchema","true").mode("append").saveAsTable(SNAPSHOT_TABLE)
    msg.append(f"appended {TOTAL} rows to `{SNAPSHOT_TABLE}` (snapshot_id={SNAP_ID})")
else:
    msg.append(f"WRITE_TABLE=False -> not written to `{SNAPSHOT_TABLE}`")

folder=None
if WRITE_FILES:
    user=spark.sql("SELECT current_user()").first()[0]; ts=NOW.strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/segmentation_baseline/{DATA_CUT_LABEL}_{ts}"
    dbutils.fs.mkdirs(f"file:{folder}")
    pdf=snap.orderBy("assigned_state","appealReferenceNumber").toPandas()
    try:
        import openpyxl  # noqa
    except Exception:
        import subprocess,sys; subprocess.run([sys.executable,"-m","pip","install","-q","openpyxl"])
    xlsx=f"{folder}/segmentation_snapshot_{DATA_CUT_LABEL}.xlsx"; pdf.to_excel(xlsx, index=False)
    dbfs_parq=f"dbfs:/tmp/segmentation_baseline/{DATA_CUT_LABEL}_{ts}.parquet"
    snap.write.mode("overwrite").parquet(dbfs_parq)
    msg.append(f"xlsx -> {xlsx}"); msg.append(f"parquet -> {dbfs_parq}")

In [0]:
# ---- CELL 3 : summary print ----
by_tier = {r["tier"]: r["n"] for r in snap.groupBy("tier").agg(count("*").alias("n")).collect()}
by_state=snap.groupBy("tier","assigned_state").agg(count("*").alias("n"),
          sum(col("is_clone").cast("int")).alias("clones"),
          sum(when(col("is_valid")==True,1).otherwise(0)).alias("valid")).orderBy("tier","assigned_state")
lines=["="*70, f"SEGMENTATION SNAPSHOT  '{DATA_CUT_LABEL}'  id={SNAP_ID}", "="*70,
       f"total cases: {TOTAL}   (active: {by_tier.get('active',0)}   archive: {by_tier.get('archive',0)})",
       f"clones: {snap.filter(col('is_clone')).count()}", "",
       f"{'tier':8s} {'state':30s} {'cases':>7} {'valid':>6} {'clones':>7}", "-"*62]
for r in by_state.collect():
    lines.append(f"{r['tier']:8s} {r['assigned_state']:30s} {r['n']:>7} {r['valid'] or 0:>6} {r['clones'] or 0:>7}")
lines+=["-"*62]+msg
if skipped: lines+=["", "NOTES:"]+[f"  - {s[0]}: {s[1]}" for s in skipped]
print("\n".join(lines))